# Anotador generico via notebook

Use este notebook quando quiser anotar direto pelo Jupyter, sem abrir o app Streamlit. Ele usa a mesma estrutura de projeto em `annotations/`, salva labels YOLO e permite exportar o dataset no final.

A janela de anotacao usa OpenCV GUI. Se aparecer erro de GUI, instale `opencv-python` e remova `opencv-python-headless` do ambiente.

## 1. Configurar projeto

Troque `PROJECT_NAME`, `IMAGE_DIR`, `CLASSES` e `INSTRUCTIONS` para qualquer tarefa: caixa, avaria, onibus quebrado, numero de onibus ou outro objeto.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebook':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from annotation_app.project_io import create_or_update_project, import_legacy_yolo_labels, load_image_index, progress
from annotation_app.notebook_annotator import run_notebook_annotator
from annotation_app.exporter import export_yolo

BASES_DIR = PROJECT_ROOT / 'bases'
PROJECTS_ROOT = PROJECT_ROOT / 'annotations'
PROJECTS_ROOT.mkdir(exist_ok=True)

# Exemplos:
# PROJECT_NAME = 'numero_onibus'; IMAGE_DIR = BASES_DIR / 'numero_onibus'; CLASSES = [{'id': 0, 'name': 'NUMERO_ONIBUS', 'color': '#00c8ff'}]
# PROJECT_NAME = 'caixas'; IMAGE_DIR = BASES_DIR / 'caixas'; CLASSES = [{'id': 0, 'name': 'CAIXA', 'color': '#f59e0b'}]
# PROJECT_NAME = 'avarias'; IMAGE_DIR = BASES_DIR / 'avarias'; CLASSES = [{'id': 0, 'name': 'AVARIA', 'color': '#ef4444'}]

PROJECT_NAME = 'meu_projeto'
IMAGE_DIR = BASES_DIR / 'minhas_imagens'
CLASSES = [
    {'id': 0, 'name': 'OBJETO', 'color': '#00c8ff'},
]
INSTRUCTIONS = 'Descreva aqui exatamente o que deve ser anotado e o que deve ser ignorado.'

project_path = create_or_update_project(PROJECTS_ROOT, PROJECT_NAME, str(IMAGE_DIR), CLASSES, INSTRUCTIONS)
print(f'Projeto pronto: {project_path}')
print(f'Imagens indexadas: {len(load_image_index(project_path))}')

Projeto pronto: d:\Projetos\local-vision-annotator\annotations\meu_projeto
Imagens indexadas: 0


## 2. Rodar anotador no notebook

Controles da janela:

- Mouse: desenhar box
- `ENTER`: salvar imagem como anotada e ir para a proxima
- `D`: marcar como sem objeto
- `R`: revisar depois
- `S`: pular
- `Z`: desfazer ultima box
- `C`: alternar classe
- `0-9`: escolher classe pelo id
- `N` / `P`: proxima / anterior
- `ESC`: sair

In [ ]:
run_notebook_annotator(
    project_path,
    status_filter=('pending', 'needs_review'),
    reannotate=False,
    max_width=1280,
    max_height=820,
)

## 3. Importar labels YOLO antigas, se precisar

Use quando ja existir uma pasta de labels antiga, por exemplo `labels_numero/`. A importacao tenta casar labels com imagens pelo nome do arquivo.

In [ ]:
LEGACY_LABELS_DIR = IMAGE_DIR / 'labels_numero'
LEGACY_CLASS_ID = 0

if LEGACY_LABELS_DIR.exists():
    result = import_legacy_yolo_labels(project_path, LEGACY_LABELS_DIR, class_id=LEGACY_CLASS_ID)
    print(result)
else:
    print(f'Pasta nao encontrada: {LEGACY_LABELS_DIR}')

## 4. Ver progresso

In [ ]:
images = load_image_index(project_path)
print(progress(project_path, images))

## 5. Exportar dataset YOLO

A exportacao copia imagens e labels para `annotations/<projeto>/exports/...` e gera `data.yaml`.

In [ ]:
export_dir = export_yolo(
    project_path,
    train_pct=80,
    val_pct=15,
    seed=42,
    include_empty=True,
)
print(f'Dataset exportado em: {export_dir}')